In [67]:
import pandas as pd
import os
import glob
import datetime
from collections import Counter

IMPORTS_DIR = "imports/"
PROCESSED_DIR = "processed/"
ARCHIVES_DIR = "archives/"

PassCounter = 0
FailCounter = 0
RemoveCounter = 0

files = glob.glob(os.path.join(IMPORTS_DIR, "**", "*.xls*"), recursive=True)

column_map = {
    "matter nr": "matter no",
    "matter no": "matter no",
    "matter number": "matter no",
    "matter to": "matter no",
    "ref no": "matter no",
    "matter": "matter no",
    "" : "matter no",
}

unwanted_patterns = ["999999", "FULL BANK STATEMENT", "TRANSACTIONS FOR TRUST ACCOUNT"]

required_cols = ["date", "description", "matter no", "amount"]

def fix_date_format(x):
    if isinstance(x, (pd.Timestamp, datetime.date)):
        return pd.to_datetime(x)
    x = str(x).strip()
    if "/" in x:
        return pd.to_datetime(x, errors="coerce")
    elif len(x) == 8 and x.isdigit():
        return pd.to_datetime(f"{x[0:4]}/{x[4:6]}/{x[6:8]}", errors="coerce")
    return pd.NaT

for file in files:
    try:
        filename = os.path.basename(file)

        # 🔎 Delete unwanted files first
        if (
            filename.startswith("~$") or
            any(pat in filename.upper() for pat in unwanted_patterns) or
            filename.endswith("(2).xlsx") or filename.endswith("(2).xls") or filename.endswith("(2).xlsm")
        ):
            os.remove(file)
            RemoveCounter += 1
            continue

        df = pd.read_excel(file)
        df.columns = df.columns.str.lower().str.strip()
        df.rename(columns=column_map, inplace=True)
        df = df[required_cols]
        df = df[df['amount'] >= 0]
        df["allocated"] = df["matter no"].apply(lambda x: "1" if x != 999999 else "0")
        df["date"] = df["date"].apply(fix_date_format).dt.strftime("%Y-%m-%d")

        rel_path = os.path.relpath(file, IMPORTS_DIR)
        processed_path = os.path.join(PROCESSED_DIR, rel_path)
        processed_path = os.path.splitext(processed_path)[0] + ".csv"
        os.makedirs(os.path.dirname(processed_path), exist_ok=True)

        df.to_csv(processed_path, index=False)
        os.remove(file)

        folder = os.path.dirname(file)
        try:
            os.removedirs(folder)
            print(f"Deleted empty folder: {folder}")
        except OSError:
            pass

        PassCounter += 1

    except Exception as e:
        print(f"\033[91mError processing file {file}: {e}\033[0m")
        FailCounter += 1

if PassCounter == 0 and RemoveCounter == 0 and FailCounter == 0:
    print("\033[93mNo files found to process.\033[0m")
else:
    print(f"\033[92m{PassCounter} files successfully processed.\033[0m")
    print(f"\033[93m{RemoveCounter} unwanted files removed.\033[0m")
    print(f"\033[91m{FailCounter} files failed to process.\033[0m")


128 files successfully processed.
16 unwanted files removed.
0 files failed to process.
